# Explainable Used-Car Price Prediction with CarDekho

This notebook implements a complete tabular regression workflow for predicting used-car selling prices.

### What it covers
- data cleaning and validation
- numerical scaling and categorical one-hot encoding
- Linear Regression baseline
- Random Forest regression
- Deep Neural Network regression
- Random Forest + PCA feature augmentation
- Random Forest + K-Means feature augmentation
- MAE, RMSE, R², training time and inference time
- feature-importance analysis

The repository includes the completed experiment outputs under `results/`, so the trained model comparisons and visualizations can be reviewed immediately. The notebook also contains the full training workflow for reproducibility.

## 1. Imports and configuration

This project uses Python, pandas, NumPy, scikit-learn, TensorFlow/Keras and plotting libraries.  
This version groups the imports and configuration more cleanly so the workflow is easier to maintain.

In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
TEST_SIZE = 0.20
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

## 2. Load and inspect the CarDekho dataset

In [ ]:
DATA_PATH = Path("../data/cardekho_dataset.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/cardekho_dataset.csv")

cars = pd.read_csv(DATA_PATH)
print("Raw shape:", cars.shape)
display(cars.head())

## 3. Data cleaning

In [ ]:
def clean_cardekho_data(frame: pd.DataFrame) -> pd.DataFrame:
    cleaned = frame.copy()

    # Remove exact duplicate listings.
    cleaned = cleaned.drop_duplicates()

    # Remove the exported row-index column when present.
    for column in ["Unnamed: 0", "index"]:
        if column in cleaned.columns:
            cleaned = cleaned.drop(columns=column)

    return cleaned

cars = clean_cardekho_data(cars)

print("Cleaned shape:", cars.shape)
print("\nMissing values:")
display(cars.isna().sum().to_frame("missing"))

## 4. Define target and predictor groups

`selling_price` is the regression target.

The model uses a mixture of numerical and categorical predictors.  
`car_name` is intentionally excluded because `brand` and `model` already capture the main identity information while avoiding a second highly redundant text-like category.

In [ ]:
TARGET = "selling_price"

numeric_features = [
    "vehicle_age",
    "km_driven",
    "mileage",
    "engine",
    "max_power",
    "seats",
]

categorical_features = [
    "brand",
    "model",
    "seller_type",
    "fuel_type",
    "transmission_type",
]

feature_columns = numeric_features + categorical_features

X = cars[feature_columns].copy()
y = cars[TARGET].copy()

print("Feature matrix:", X.shape)
print("Target vector:", y.shape)

## 5. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 6. Shared preprocessing

Numerical columns are standardized with `StandardScaler`.  
Categorical columns are converted to one-hot vectors.  
`handle_unknown="ignore"` prevents unseen categories at test/inference time from crashing the transformation.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(drop="first", handle_unknown="ignore"),
            categorical_features,
        ),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

## 7. Evaluation helper

All models are evaluated using the same metrics:
- **MAE**: mean absolute prediction error
- **RMSE**: penalizes large prediction errors more strongly
- **R²**: proportion of target variance explained by the model

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

def fit_sklearn_model(name, estimator, x_train, y_train, x_test, y_test):
    start = time.perf_counter()
    estimator.fit(x_train, y_train)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    predictions = estimator.predict(x_test)
    inference_time = time.perf_counter() - start

    row = {"Model": name, **regression_metrics(y_test, predictions)}
    row["Training Time (s)"] = train_time
    row["Inference Time (s)"] = inference_time
    return estimator, predictions, row

## 8. Linear Regression

In [ ]:
linear_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("regressor", LinearRegression()),
    ]
)

# Training:
# linear_model, linear_predictions, linear_result = fit_sklearn_model(
#     "Linear Regression", linear_model, X_train, y_train, X_test, y_test
# )

## 9. Random Forest

In [ ]:
random_forest_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=100,
                max_depth=10,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

# Training:
# random_forest_model, rf_predictions, rf_result = fit_sklearn_model(
#     "Random Forest", random_forest_model, X_train, y_train, X_test, y_test
# )

## 10. Deep Neural Network

In [ ]:
# Neural networks work with a dense matrix.
X_train_dense = (
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else np.asarray(X_train_processed)
)
X_test_dense = (
    X_test_processed.toarray()
    if hasattr(X_test_processed, "toarray")
    else np.asarray(X_test_processed)
)

def build_dnn(input_size: int) -> keras.Model:
    network = keras.Sequential(
        [
            layers.Input(shape=(input_size,)),
            layers.Dense(64, activation="relu"),
            layers.Dense(32, activation="relu"),
            layers.Dense(1),
        ]
    )
    network.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return network

dnn_model = build_dnn(X_train_dense.shape[1])

# Training:
# start = time.perf_counter()
# history = dnn_model.fit(
#     X_train_dense,
#     y_train,
#     validation_split=0.2,
#     epochs=50,
#     batch_size=32,
#     verbose=0,
# )
# dnn_train_time = time.perf_counter() - start
#
# start = time.perf_counter()
# dnn_predictions = dnn_model.predict(X_test_dense, verbose=0).ravel()
# dnn_inference_time = time.perf_counter() - start

## 11. Random Forest + PCA

Five principal components are learned from the processed feature matrix and appended to the original processed features.  
The augmented representation is then passed to another Random Forest regressor.

In [ ]:
pca = PCA(n_components=5, random_state=RANDOM_STATE)

# PCA requires dense input.
train_dense_for_pca = (
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else np.asarray(X_train_processed)
)
test_dense_for_pca = (
    X_test_processed.toarray()
    if hasattr(X_test_processed, "toarray")
    else np.asarray(X_test_processed)
)

pca_train = pca.fit_transform(train_dense_for_pca)
pca_test = pca.transform(test_dense_for_pca)

X_train_pca_augmented = np.hstack([train_dense_for_pca, pca_train])
X_test_pca_augmented = np.hstack([test_dense_for_pca, pca_test])

rf_pca = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# Training:
# rf_pca, rf_pca_predictions, rf_pca_result = fit_sklearn_model(
#     "RF + PCA",
#     rf_pca,
#     X_train_pca_augmented,
#     y_train,
#     X_test_pca_augmented,
#     y_test,
# )

## 12. Random Forest + K-Means

K-Means discovers three latent vehicle groups from the processed training features.  
The resulting cluster label is appended as an additional feature before Random Forest training.

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=RANDOM_STATE,
    n_init=10,
)

train_clusters = kmeans.fit_predict(X_train_dense).reshape(-1, 1)
test_clusters = kmeans.predict(X_test_dense).reshape(-1, 1)

X_train_cluster_augmented = np.hstack([X_train_dense, train_clusters])
X_test_cluster_augmented = np.hstack([X_test_dense, test_clusters])

rf_kmeans = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# Training:
# rf_kmeans, rf_kmeans_predictions, rf_kmeans_result = fit_sklearn_model(
#     "RF + KMeans",
#     rf_kmeans,
#     X_train_cluster_augmented,
#     y_train,
#     X_test_cluster_augmented,
#     y_test,
# )

## 13. Stored experimental results

The repository includes the completed experiment results so the portfolio can be inspected without re-running model training.  
These values summarize the completed model evaluation.

In [ ]:
RESULTS_PATH = Path("../results/model_results.csv")
if not RESULTS_PATH.exists():
    RESULTS_PATH = Path("results/model_results.csv")

results = pd.read_csv(RESULTS_PATH)
display(results)

### Result summary

The experiment shows that **Random Forest + K-Means** achieved the strongest overall predictive performance:

- lowest MAE
- lowest RMSE
- highest R²

The DNN required substantially more training time while producing a lower R² than the tree-based models.

## 14. Visualize stored results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
palette = ["gold", "goldenrod", "khaki", "darkgoldenrod", "palegoldenrod"]

axes[0].bar(results["Model"], results["MAE"], color=palette)
axes[0].set_title("Average Absolute Error by Model")
axes[0].set_ylabel("MAE")

axes[1].bar(results["Model"], results["RMSE"], color=palette)
axes[1].set_title("Root Mean Squared Error by Model")
axes[1].set_ylabel("RMSE")

axes[2].bar(results["Model"], results["R2"], color=palette)
axes[2].set_title("Explained Variance by Model")
axes[2].set_ylabel("R²")

for ax in axes:
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 15. Feature importance

When the `RF + KMeans` training cell is executed, feature importance can be extracted from the trained forest.  
A stored importance plot is already available in `results/figures/feature_importance.png`.

In [ ]:
# Run this after fitting `rf_kmeans`.
#
# encoded_names = list(
#     preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features)
# )
# feature_names = numeric_features + encoded_names + ["cluster_label"]
#
# importance = pd.Series(
#     rf_kmeans.feature_importances_,
#     index=feature_names,
# ).sort_values(ascending=False)
#
# display(importance.head(10))
# importance.head(10).sort_values().plot.barh(figsize=(8, 5))
# plt.title("Top 10 Feature Importances — Random Forest + K-Means")
# plt.xlabel("Importance")
# plt.tight_layout()
# plt.show()

## 16. Conclusions

This experiment illustrates three useful ML observations:

1. A simple linear model provides a useful baseline but struggles with nonlinear price relationships.
2. Random Forest captures tabular feature interactions much more effectively.
3. Adding a K-Means cluster feature can provide extra latent segmentation and, in this experiment, produced the strongest metrics.

### Limitations
- results are based on a single CarDekho dataset
- geographic market variation is not explicitly modeled
- rare brands/models can be underrepresented
- feature importance is global and does not explain each individual prediction

### Possible extensions
- SHAP explanations
- gradient boosting/XGBoost
- cross-validation and systematic hyperparameter tuning
- model persistence and a small web prediction interface